In [ ]:
# Este código se modificó para que el modelo use edge_weight proveniente del DAta

2.6.0+cu124
12.4


In [ ]:
# Torch version
!python -c "import torch; print(torch.__version__)"

# Cuda version
!python -c "import torch; print(torch.version.cuda)"

In [2]:
import torch
torch.cuda.empty_cache()

In [3]:
# Uninstall
# !pip uninstall torch-scatter torch-sparse torch-cluster torch-spline-conv pyg-lib -y

In [4]:
# Update Torch
# !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [5]:
# Install PyG (automatic)
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{torch.__version__}.html
# !pip install torch_geometric

In [6]:
# Verify instalation
import torch
import torch_geometric
import torch_scatter

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch_scatter.__version__)
print(torch_geometric.__version__)

2.6.0+cu124
12.4
True
2.1.2+pt26cu124
2.7.0


In [7]:
from model_PyG_cla import *
from utils import *

In [8]:
import json
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import time
import torch
import torch.nn.functional as F
import torch_geometric.transforms as T

from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import InMemoryDataset, Data
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import Compose
from torch_geometric.utils import dense_to_sparse, negative_sampling
from torch.nn.functional import binary_cross_entropy_with_logits
from torch.optim import Adam

In [9]:
import torch_geometric
print(torch_geometric.__version__)

2.7.0


### Utils

In [10]:
""" def info(data):
	print("Validate:\t {}".format(data.validate(raise_on_error=True)))
	print("Num. nodes:\t {}".format(data.num_nodes))
	print("Num. edges:\t {}".format(data.num_edges))
	print("Num. features:\t {}".format(data.num_node_features))
	print("Has isolated:\t {}".format(data.has_isolated_nodes()))
	print("Has loops:\t {}".format(data.has_self_loops()))
	print("Is directed:\t {}".format(data.is_directed()))
	print("Is undirected:\t {}".format(data.is_undirected()))
	print("{}".format(data.edge_index))
	print("{}".format(data.x))
	print("{}".format(data.edge_attr))

def compute_num_neg_samples(edge_index, num_nodes, ratio):
	E = edge_index.size(1)
	max_neg = num_nodes * num_nodes - E
	return min(int(ratio * E), max_neg)

def neg_ratio_schedule(epoch, max_epoch):
	start = 5.0
	end = 1.0
	return start - (start - end) * (epoch / max_epoch)

class EarlyStopping:
	def __init__(self, patience=5, delta=0, warmup=5, verbose=False):
		self.patience = patience
		self.delta = delta
		self.warmup = warmup
		self.verbose = verbose
		self.best_loss = None
		self.no_improvement_count = 0
		self.stop_training = False
	
	def check_early_stop(self, loss, epoch):
		if epoch >= self.warmup:
			if self.best_loss is None or loss < self.best_loss - self.delta:
				self.best_loss = loss
				self.no_improvement_count = 0
			else:
				self.no_improvement_count += 1
				if self.no_improvement_count >= self.patience:
					self.stop_training = True
					if self.verbose:
						print("Stopping early as no improvement has been observed.") """

' def info(data):\n\tprint("Validate:\t {}".format(data.validate(raise_on_error=True)))\n\tprint("Num. nodes:\t {}".format(data.num_nodes))\n\tprint("Num. edges:\t {}".format(data.num_edges))\n\tprint("Num. features:\t {}".format(data.num_node_features))\n\tprint("Has isolated:\t {}".format(data.has_isolated_nodes()))\n\tprint("Has loops:\t {}".format(data.has_self_loops()))\n\tprint("Is directed:\t {}".format(data.is_directed()))\n\tprint("Is undirected:\t {}".format(data.is_undirected()))\n\tprint("{}".format(data.edge_index))\n\tprint("{}".format(data.x))\n\tprint("{}".format(data.edge_attr))\n\ndef compute_num_neg_samples(edge_index, num_nodes, ratio):\n\tE = edge_index.size(1)\n\tmax_neg = num_nodes * num_nodes - E\n\treturn min(int(ratio * E), max_neg)\n\ndef neg_ratio_schedule(epoch, max_epoch):\n\tstart = 5.0\n\tend = 1.0\n\treturn start - (start - end) * (epoch / max_epoch)\n\nclass EarlyStopping:\n\tdef __init__(self, patience=5, delta=0, warmup=5, verbose=False):\n\t\tself.p

### Parameters

In [11]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"] # Change to static, e.g. "exp1"

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

raw_data_file = params["raw_data_file"]
print("Raw data:\t", raw_data_file)

methods = params["methods"]
print("Methods:\t", methods)

apply_transformation = params["apply_transformation"]
print("Apply transformation:", apply_transformation)

dimension = params["dimension"]
print("Dimension:\t", dimension)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

cuda = params["cuda"]
print("Cuda:\t\t", cuda)

epochs = params["epochs"]
print("Epochs:\t\t", epochs)

lr = params["lr"]
print("Lr:\t\t", lr)

weight_decay = params["weight_decay"]
print("Weight decay:\t", weight_decay)

patience = params["patience"]
print("Patience:\t", patience)

Exp:		 exp110
Raw data:	 Vanessa_september_2br_10ar_format
Methods:	 ['t-gae']
Apply transformation: True
Dimension:	 128
Groups id:	 ['SecoAmazonas', 'FrescoAmazonas']
Subgroups id:	 {'SecoAmazonas': ['1', '2'], 'FrescoAmazonas': ['1', '2']}
Cuda:		 1
Epochs:		 500
Lr:		 0.0001
Weight decay:	 0.0005
Patience:	 20


In [12]:
# Remove
# groups_id = ["LSNB", "BC", "RCC", "BPH", "PD"] # "OSA" Memory issue
# groups_id = ["CKD-Severe", "LPRD"] # ["AR", "Health", "CKD-Mild", "CKD-Severe", "LPRD"]

### Setup

In [13]:
# Parameters models

dataset = exp
encoders = ["GCNRes", "GIN", "GINE"] # ["GIN", "GINE"] # ["GCNRes", "GCN", "GIN", "GAT", "GATE", "GINE"] # ["GIN", "GINE"] # ["GIN", "GINE"] # ["GIN", "GINE"] # Change
device = torch.device(f"cuda:{cuda}" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

NUM_HIDDEN_LAYERS = 12
HIDDEN_DIM = [1024] * NUM_HIDDEN_LAYERS + [1024]
output_feature_size = dimension # 128
# lr = 0.001
# epochs = 100

In [14]:
# Create datasets (groups and subgroups)

list_train_set = []

for group_id in groups_id:
	train_set = []
	for subgroup_id in subgroups_id[group_id]:
		train_set.append("{}_{}".format(group_id, subgroup_id))
	list_train_set.append(train_set)
list_train_set

[['SecoAmazonas_1', 'SecoAmazonas_2'],
 ['FrescoAmazonas_1', 'FrescoAmazonas_2']]

### Create Data (PyG)

In [15]:
# Only for GIN
""" transform = Compose([
	# T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	T.AddSelfLoops(fill_value=1.0),
	T.ToDevice(device)
]) """

# For GIN and GINE
transform = T.Compose([
	T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	# T.AddSelfLoops(attr="edge_attr", fill_value="mean"),
	# T.AddLaplacianEigenvectorPE(k=3, attr_name=None, is_undirected=True),
	# T.AddRandomWalkPE(walk_length=8, attr_name=None),
	T.ToDevice(device)
])

In [16]:
print("Loading training datasets")

list_train_loader = []
# scaler = StandardScaler()

features = [0, 1, 2, 4, 5] # Change
# features = [0, 2, 4, 5] # Change (for Pablo)

for group_id in groups_id:
	train_loader = {}
	for subgroup_id in subgroups_id[group_id]:
		df_nodes = pd.read_csv(f"experiments/output/{exp}/preprocessing/graphs_data/nodes_{group_id}_{subgroup_id}.csv")
		# idx, id, mz, rt, 0, 1, 2, ...

		df_intensity = df_nodes.iloc[:, 4:]
		# 0, 1, 2, ...

		df_edges = pd.read_csv(f"experiments/output/{exp}/preprocessing/graphs_data/edges_{group_id}_{subgroup_id}.csv")
		# source, target, weight, subgroup

		# Node features
		""" mz = np.log10(df_nodes["mz"].values + 1e-8) # Log-transform m/z to stabilize scale differences
		rt = df_nodes["rt"].values
		rt = (rt - rt.mean()) / (rt.std() + 1e-8) # Z-score normalization for retention time (RT)

		intensity = df_intensity.values.astype(np.float32)
		intensity = intensity / (intensity.sum(axis=0, keepdims=True) + 1e-8)
		intensity_mean = intensity.mean(axis=1)

		intensity_std = intensity.std(axis=1)
		intensity_cv = intensity_std / (intensity_mean + 1e-8)
		intensity_cv = np.log1p(np.clip(intensity_cv, 0, 2.0))

		presence_ratio = (df_intensity > 0).mean(axis=1) # Acts as a reliability / confidence signal """

		# df_intensity = np.sign(df_intensity) * np.log10(np.abs(df_intensity) + 1e-8)

		mz = np.log10(df_nodes.iloc[:, 2].values + 1e-8)
		rt = df_nodes.iloc[:, 3].values
		rt = (rt - rt.mean()) / (rt.std() + 1e-8)
		intensity_mean = df_intensity.mean(axis=1).values
		intensity_std = df_intensity.std(axis=1).values
		intensity_cv = intensity_std / (intensity_mean + 1e-8)
		# presence_ratio = (df_intensity > 0).mean(axis=1)
		threshold = df_intensity.quantile(0.2).mean() # Change value
		presence_ratio = (df_intensity > threshold).mean(axis=1)

		x = np.stack([
				mz,             				# 0 physicochemical identity
				rt,								# 1 chromatographic alignment
				# np.log10(intensity_mean + 1e-8),# 2 global abundance
				intensity_mean,					# 2 global abundance
				intensity_std,  				# 3
				intensity_cv,   				# 4 robustness (anti-oversmoothing)
				presence_ratio  				# 5 reliability
			], axis=1)

		x = torch.tensor(x[:, features], dtype=torch.float) # [N, F]
		# x = torch.tensor(scaler.fit_transform(x.numpy())) # scaling

		# Edge index
		edge_index = torch.tensor(df_edges.iloc[:, [0, 1]].values.T, dtype=torch.long) # [2, E]

		# Edge attribute
		edge_weight = torch.tensor(df_edges.iloc[:, 2].values, dtype=torch.float) # [E]

		# edge_attr = edge_weight.view(-1, 1) # [E,1]
		edge_attr = torch.stack([
							edge_weight.abs(),        # strength
							torch.sign(edge_weight),  # direction (+1, -1)
							edge_weight ** 2          # nonlinearity
						], dim=1) # [E, 3]

		# Reduce number of edges
		""" mask = torch.abs(edge_weight) > 0.995
		edge_index = edge_index[:, mask]
		edge_attr  = edge_attr[mask] """

		data = Data(x=x, edge_index=edge_index.contiguous(), edge_attr=edge_attr, edge_weight=edge_weight.abs())

		data = transform(data)
		# data = data.to(device)

		train_loader[f"{group_id}_{subgroup_id}"] = data
		print(data)
		info_data(data, include_vectors=True)

		df_node_alignment_truth = None # No use
	list_train_loader.append(train_loader)

Loading training datasets
Data(x=[1570, 5], edge_index=[2, 303748], edge_attr=[303748, 3], edge_weight=[303748])
Validate:	 True
Num. nodes:	 1570
Num. edges:	 303748
Num. features:	 5
Has isolated:	 False
Has loops:	 False
Is directed:	 False
Is undirected:	 True
tensor([[   0,    0,    0,  ..., 1569, 1569, 1569],
        [   1,    2,    3,  ..., 1555, 1556, 1557]], device='cuda:1')
tensor([[3.1269e-06, 3.5459e-07, 9.9999e-01, 1.2675e-06, 2.1236e-06],
        [1.3104e-06, 7.5462e-07, 1.0000e+00, 5.6884e-07, 8.8887e-07],
        [1.7239e-06, 2.1731e-07, 1.0000e+00, 6.8093e-07, 1.1635e-06],
        ...,
        [1.0479e-04, 1.0935e-05, 9.9977e-01, 7.3078e-05, 3.9521e-05],
        [2.3508e-05, 7.6022e-06, 9.9994e-01, 1.2824e-05, 1.2167e-05],
        [6.0799e-06, 6.3445e-07, 9.9999e-01, 3.4013e-06, 3.3948e-06]],
       device='cuda:1')
tensor([[ 0.7046,  1.0000,  0.4965],
        [ 0.8530, -1.0000,  0.7275],
        [ 0.7125, -1.0000,  0.5076],
        ...,
        [ 0.6341, -1.0000,  0.4

In [17]:
df_node_alignment_truth

In [18]:
# Important

list_ = list(range(data.num_nodes))
df_node_alignment_truth = pd.DataFrame([list_, list_])
df_node_alignment_truth = df_node_alignment_truth.T + 1
df_node_alignment_truth

,0,1
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5
...,...,...
1565,1566,1566
1566,1567,1567
1567,1568,1568
1568,1569,1569


### Train

In [19]:
"""Entrenamiento T-GAE para subgraph alignment (PyTorch Geometric).

Requiere el modelo refactorizado de model_PyG.py (edge_weight / edge_attr por nombre).

Cambios respecto a la versión anterior de fit_TGAE_subgraph:
  1. Los positivos de la reconstrucción son las aristas ORIGINALES sin self-loops
	 (data.pos_edge_index), no data.edge_index (que trae self-loops de AddSelfLoops).
  2. Selección de modelo opcional por alineamiento en un conjunto de validación
	 (val_pairs) y restauración de los mejores pesos. Sin val_pairs: early stopping
	 por pérdida, como antes.
  3. Decoder opcional "cosine" (coseno / tau + sesgo aprendible) además del producto interno.
  4. Aumentación opcional por eliminación de aristas (aug_p > 0): se codifica el grafo
	 perturbado y se reconstruye el grafo original (Ec. 7 del paper, aplicada por grafo).
  5. Pérdida balanceada: 0.5 * (BCE positivos + BCE negativos), con neg_ratio configurable.
  6. Opcional: recorte de gradiente, AdamW, semilla.
  7. Métricas de alineamiento (Hit@K, MRR) con similitud coseno, L2 o CSLS.
"""
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.utils import negative_sampling, remove_self_loops
from tqdm import tqdm


GCNRES_NAMES = ("GCNRes", "ORI")  # "ORI" es el nombre que usa el notebook para la capa original

# ----------------------------------------------------------------------------
# Utilidades
# ----------------------------------------------------------------------------
def set_seed(seed):
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)


def make_val_split(pairs, val_frac=0.2, seed=0):
	"""Divide pares de anclaje [K, 2] en (validación, evaluación).

	Reporta las métricas finales SOLO sobre el segundo conjunto.
	"""
	pairs = np.asarray(pairs)
	idx = np.random.default_rng(seed).permutation(len(pairs))
	n_val = int(round(val_frac * len(pairs)))
	return pairs[idx[:n_val]], pairs[idx[n_val:]]


def prepare_graph_v1(data):
	"""Guarda las aristas originales sin self-loops (objetivo de reconstrucción)."""
	if getattr(data, "pos_edge_index", None) is None:
		data.pos_edge_index, _ = remove_self_loops(data.edge_index)
	return data


def drop_undirected_edges(edge_index, edge_attr, p, num_nodes):
	"""Elimina cada arista no dirigida con probabilidad p (ambos sentidos a la vez).

	Conserva los self-loops y mantiene edge_attr alineado con edge_index.
	"""
	row, col = edge_index
	key = torch.minimum(row, col) * num_nodes + torch.maximum(row, col)
	uniq, inv = torch.unique(key, return_inverse=True)
	keep_undirected = torch.rand(uniq.numel(), device=edge_index.device) >= p
	keep = keep_undirected[inv] | (row == col)
	edge_index = edge_index[:, keep]
	if edge_attr is not None:
		edge_attr = edge_attr[keep]
	return edge_index, edge_attr


def _encode_v1(model, data, encoder, aug_p=0.0):
	"""Llama al modelo según el encoder. GCNRes recibe aristas sin loops y el modelo
	calcula Â = D^-1/2 (A+I) D^-1/2 internamente; GCN recibe pesos crudos (ninguno)."""
	ei = data.pos_edge_index if encoder in GCNRES_NAMES else data.edge_index
	ea = data.edge_attr if encoder in ("GINE", "GATE") else None
	if aug_p > 0:
		ei, ea = drop_undirected_edges(ei, ea, aug_p, data.num_nodes)
	kw = {"edge_attr": ea} if ea is not None else {}
	return model(data.x, ei, **kw)

def undirected_keep_mask(edge_index, p, num_nodes):
    """Boolean mask: drop each undirected edge with probability p (both directions together); loops are kept."""
    row, col = edge_index
    key = torch.minimum(row, col) * num_nodes + torch.maximum(row, col)
    uniq, inv = torch.unique(key, return_inverse=True)
    keep_undirected = torch.rand(uniq.numel(), device=edge_index.device) >= p
    return keep_undirected[inv] | (row == col)


def prepare_graph(data):
    if getattr(data, "pos_edge_index", None) is None:
        ew = getattr(data, "edge_weight", None)
        # weights follow the edges when self-loops are removed
        data.pos_edge_index, data.pos_edge_weight = remove_self_loops(data.edge_index, ew)
    return data


def _encode(model, data, encoder, aug_p=0.0):
    use_ori = encoder in GCNRES_NAMES
    ei = data.pos_edge_index if use_ori else data.edge_index
    ew = getattr(data, "pos_edge_weight" if use_ori else "edge_weight", None) if (use_ori or encoder == "GCN") else None
    ea = data.edge_attr if encoder in ("GINE", "GATE") else None
    if aug_p > 0:
        keep = undirected_keep_mask(ei, aug_p, data.num_nodes)
        ei = ei[:, keep]
        ew = ew[keep] if ew is not None else None
        ea = ea[keep] if ea is not None else None
    if use_ori and ew is not None:
        # GCNRes expects an already normalized A_hat: normalize AFTER augmentation, with self-loops
        ei, ew = gcn_norm(ei, ew, data.num_nodes, add_self_loops=True)
    kw = {}
    if ew is not None:
        kw["edge_weight"] = ew
    if ea is not None:
        kw["edge_attr"] = ea
    return model(data.x, ei, **kw)


def _edge_logits(z, pos, neg, decoder, tau, bias):
	if decoder == "cosine":
		z = F.normalize(z, dim=1)
	pos_logits = (z[pos[0]] * z[pos[1]]).sum(dim=1)
	neg_logits = (z[neg[0]] * z[neg[1]]).sum(dim=1)
	if decoder == "cosine":
		pos_logits = pos_logits / tau + bias
		neg_logits = neg_logits / tau + bias
	return pos_logits, neg_logits


# ----------------------------------------------------------------------------
# Evaluación de alineamiento
# ----------------------------------------------------------------------------
def _mean_topk(a, b, k, chunk=2048):
	"""Para cada fila de a: media de las k mayores similitudes coseno con las filas de b."""
	out = []
	for i in range(0, a.size(0), chunk):
		out.append((a[i:i + chunk] @ b.T).topk(k, dim=1).values.mean(dim=1))
	return torch.cat(out)


@torch.no_grad()
def similarity_matrix(z1, z2, retrieval="cosine", csls_k=None):
    """Full similarity matrix [N1, N2] (higher = more similar), as PlanetAlign expects."""
    if retrieval == "cosine":
        a, b = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
        sim = a @ b.T
        if csls_k:
            r_s = _mean_topk(a, b, csls_k)
            r_t = _mean_topk(b, a, csls_k)
            sim = 2 * sim - r_s.unsqueeze(1) - r_t.unsqueeze(0)
        return sim
    if retrieval == "l2":
        if csls_k:
            raise ValueError("csls_k is only supported with retrieval='cosine'")
        return -torch.cdist(z1, z2)  # distances -> similarities
    raise ValueError("retrieval must be 'cosine' or 'l2'")

@torch.no_grad()
def evaluate_alignment(z1, z2, pairs, retrieval="cosine", csls_k=None, ks=(1, 5, 10, 50), mode="ltr"):
    """Hit@K and MRR computed with PlanetAlign's hits_ks_scores and mrr_score.

    pairs: [K, 2] (idx in graph 1, idx in graph 2).
    mode: 'ltr' (graph1->graph2), 'rtl', 'mean' or 'max', as in PlanetAlign.
    Returns {"hit@k": ..., "mrr": ...}.
    """
    sim = similarity_matrix(z1, z2, retrieval, csls_k)
    pairs = torch.as_tensor(pairs, dtype=torch.long, device=sim.device)
    hits = hits_ks_scores(sim, pairs, ks=ks, mode=mode)
    out = {f"hit@{k}": hits[k] for k in ks}
    out["mrr"] = mrr_score(sim, pairs, mode=mode)
    return out
 
 
def evaluate_alignment_mean(z1, z2, pairs, **kw):
    """evaluate_alignment with mode="mean" (average of both directions, as in the notebook)."""
    return evaluate_alignment(z1, z2, pairs, mode="mean", **kw)


@torch.no_grad()
def evaluate_alignment_v1(z1, z2, pairs, retrieval="cosine", csls_k=None, ks=(1, 5, 10, 50)):
	"""Hit@K y MRR de z1 (grafo 1) hacia z2 (grafo 2). pairs: [K, 2] (idx en 1, idx en 2)."""
	pairs = torch.as_tensor(np.asarray(pairs), dtype=torch.long, device=z1.device)
	src, tgt = pairs[:, 0], pairs[:, 1]
	if retrieval == "cosine":
		a, b = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
		sim = a[src] @ b.T
		if csls_k:
			r_s = _mean_topk(a[src], b, csls_k)
			r_t = _mean_topk(b, a, csls_k)
			sim = 2 * sim - r_s.unsqueeze(1) - r_t.unsqueeze(0)
	elif retrieval == "l2":
		if csls_k:
			raise ValueError("csls_k solo está soportado con retrieval='cosine'")
		sim = -torch.cdist(z1[src], z2)
	else:
		raise ValueError("retrieval debe ser 'cosine' o 'l2'")
	true = sim[torch.arange(len(src), device=sim.device), tgt].unsqueeze(1)
	rank = (sim > true).sum(dim=1) + 1
	out = {f"hit@{k}": (rank <= k).float().mean().item() for k in ks}
	out["mrr"] = (1.0 / rank.float()).mean().item()
	print(1, out)
	# return out

	print(type(sim))
	print(sim)

	# Based on PlanetAlign
	hits = hits_ks_scores(sim.clone().detach(), torch.tensor(EVAL_PAIRS), ks=ks,  mode="mean")
	print(10)
	mrr = mrr_score(sim.clone().detach(), torch.tensor(EVAL_PAIRS), mode="mean")
	print(11)
	hits["mrr"] = mrr
	print(2, hits)
	return hits


def evaluate_alignment_mean_v1(z1, z2, pairs, **kw):
	"""Promedio de grafo1->grafo2 y grafo2->grafo1 (equivale a mode="mean" de PlanetAlign)."""
	pairs = np.asarray(pairs)
	a = evaluate_alignment_v1(z1, z2, pairs, **kw)
	b = evaluate_alignment_v1(z2, z1, pairs[:, ::-1].copy(), **kw)
	return {k: (a[k] + b[k]) / 2 for k in a}


# ----------------------------------------------------------------------------
# Entrenamiento
# ----------------------------------------------------------------------------
def fit_TGAE_subgraph(model, encoder, exp, group_id, subgroups_id, train_loader, train_set,
					no_samples, epochs, lr, weight_decay,
					val_pairs=None, eval_every=5, select_metric="mrr",
					retrieval="cosine", csls_k=None,
					decoder="dot", tau=0.2, neg_ratio=1.0,
					aug_p=0.0, grad_clip=None, optimizer_name="adam",
					patience=50, delta=1e-4, warmup=10, seed=None, save_outputs=True):
	"""Entrena el encoder y guarda embeddings y pérdida (mismas rutas que antes).

	no_samples se conserva por compatibilidad de firma; la pérdida se promedia
	sobre len(train_set). Devuelve un dict con el resumen del entrenamiento.
	"""
	if seed is not None:
		set_seed(seed)
	dev = next(model.parameters()).device
	if val_pairs is not None and len(subgroups_id) < 2:
		raise ValueError("La validación requiere dos subgrafos en subgroups_id")

	for ts in train_set:
		prepare_graph(train_loader[ts])
	for sg in subgroups_id:
		prepare_graph(train_loader[f"{group_id}_{sg}"])

	# Sesgo aprendible solo para el decoder coseno (sin él, la pérdida satura).
	bias = torch.zeros((), device=dev, requires_grad=True) if decoder == "cosine" else None
	param_groups = [{"params": list(model.parameters())}]
	if bias is not None:
		param_groups.append({"params": [bias], "weight_decay": 0.0})
	opt_cls = torch.optim.AdamW if optimizer_name == "adamw" else torch.optim.Adam
	optimizer = opt_cls(param_groups, lr=lr, weight_decay=weight_decay)

	use_val = val_pairs is not None
	list_loss, history = [], []
	best_score, best_epoch, best_state = -float("inf"), 0, None
	best_loss, bad_count, stopped_epoch = None, 0, None
	max_bad = max(1, patience // eval_every) if use_val else patience

	model.train()
	loop_obj = tqdm(range(1, epochs + 1))
	for epoch in loop_obj:
		loop_obj.set_description(f"Epoch: {epoch}")
		optimizer.zero_grad()
		loss = torch.zeros((), device=dev)

		for ts in train_set:
			data = train_loader[ts]
			z = _encode(model, data, encoder, aug_p)

			pos = data.pos_edge_index  # objetivo: grafo original, sin self-loops
			neg = negative_sampling(
				edge_index=pos, num_nodes=z.size(0),
				num_neg_samples=int(neg_ratio * pos.size(1)), method="sparse",
			)
			pos_logits, neg_logits = _edge_logits(z, pos, neg, decoder, tau, bias)
			loss = loss + 0.5 * (
				F.binary_cross_entropy_with_logits(pos_logits, torch.ones_like(pos_logits))
				+ F.binary_cross_entropy_with_logits(neg_logits, torch.zeros_like(neg_logits))
			)

		loss = loss / len(train_set)
		loss.backward()
		if grad_clip is not None:
			torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
		optimizer.step()

		loss_value = loss.item()
		list_loss.append(loss_value)
		loop_obj.set_postfix_str(f"Loss: {loss_value:.4f}")

		if use_val:
			if epoch % eval_every == 0 or epoch == epochs:
				model.eval()
				with torch.no_grad():
					z1 = _encode(model, train_loader[f"{group_id}_{subgroups_id[0]}"], encoder)
					z2 = _encode(model, train_loader[f"{group_id}_{subgroups_id[1]}"], encoder)
					metrics = evaluate_alignment(z1, z2, val_pairs, retrieval, csls_k)
				model.train()
				metrics["epoch"] = epoch
				history.append(metrics)
				score = metrics[select_metric]
				loop_obj.set_postfix_str(f"Loss: {loss_value:.4f} val_{select_metric}: {score:.4f}")
				if score > best_score + 1e-6:
					best_score, best_epoch, bad_count = score, epoch, 0
					# En CPU para no duplicar memoria de GPU.
					best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
				else:
					bad_count += 1
				if epoch >= warmup and bad_count >= max_bad:
					stopped_epoch = epoch
					print(f"Early stopping (validación) en la época {epoch}; mejor época: {best_epoch}")
					break
		else:
			if epoch >= warmup:
				if best_loss is None or loss_value < best_loss - delta:
					best_loss, bad_count = loss_value, 0
				else:
					bad_count += 1
					if bad_count >= max_bad:
						stopped_epoch = epoch
						print(f"Early stopping (pérdida) en la época {epoch}")
						break

	if use_val and best_state is not None:
		model.load_state_dict(best_state)
		print(f"Pesos restaurados de la época {best_epoch} (val_{select_metric}={best_score:.4f})")

	# Embeddings finales
	model.eval()
	list_df_node_embeddings = []
	with torch.no_grad():
		for subgroup_id in subgroups_id:
			data = train_loader[f"{group_id}_{subgroup_id}"]
			z = _encode(model, data, encoder)
			df = pd.DataFrame(z.cpu().numpy())
			df["subgroup_id"] = [subgroup_id] * len(df)

			# Get node ids to replace original Id (Important)
			df_nodes = pd.read_csv(f"experiments/output/{exp}/preprocessing/graphs_data/nodes_{group_id}_{subgroup_id}.csv")
			# idx,id,mz,rt,intensity_mean,intensity_cv
			df.insert(0, "Id", df_nodes["id"].values)
			# df.insert(0, "Id", range(len(df)))  # solo para datos que no son MS

			list_df_node_embeddings.append(df)
	df_node_embeddings_concat = pd.concat(list_df_node_embeddings, ignore_index=True)

	if save_outputs:
		emb_dir = f"experiments/output/{exp}/node_embeddings"
		loss_dir = f"experiments/output/{exp}/loss"
		os.makedirs(emb_dir, exist_ok=True)
		os.makedirs(loss_dir, exist_ok=True)
		df_node_embeddings_concat.to_csv(f"{emb_dir}/{encoder}_{group_id}.csv", index=False)
		np.save(f"{loss_dir}/{encoder}_{group_id}.npy", np.array(list_loss))

	if torch.cuda.is_available():
		torch.cuda.empty_cache()

	return {
		"best_epoch": best_epoch if use_val else None,
		"best_val_score": best_score if use_val else None,
		"stopped_epoch": stopped_epoch,
		"epochs_run": len(list_loss),
		"loss": list_loss,
		"val_history": history,
	}

In [20]:
# [MEJORA 1] Hiperparámetros de entrenamiento y partición de pares en un solo lugar
SEED = 42
WEIGHT_DECAY = weight_decay         # antes fijo dentro de la función
HEADS = 4                    # solo GAT / GATE

# Selección de modelo (usa anclajes -> supervisión; poner False para el protocolo no supervisado)
USE_VALIDATION = False
VAL_FRAC = 0.2
EVAL_EVERY = 5
SELECT_METRIC = "mrr"        # "mrr" | "hit@1" | "hit@5" | "hit@10" | "hit@50"
RETRIEVAL = "l2"             # "l2" (como el paper y la evaluación final) | "cosine"

# Objetivo de entrenamiento (valores por defecto = comportamiento previo / paper)
DECODER = "dot"              # "dot" | "cosine" (con TAU y sesgo aprendible)
TAU = 0.2
NEG_RATIO = 1.0              # negativos por positivo
AUG_P = 0.0                  # probabilidad de eliminar cada arista al codificar (experimento A/B)
GRAD_CLIP = None             # p. ej. 1.0

# Con validación el criterio se mide en épocas: 50 = 10 evaluaciones sin mejora con EVAL_EVERY = 5
PATIENCE = 50 if USE_VALIDATION else patience
WARMUP = 10

# Nombre en `encoders` -> tipo de convolución de model_PyG.TGAE
ENCODER_TO_CONV = {"GCNRes": "gcnres", "GIN": "gin", "GINE": "gine",
				   "GCN": "gcn", "GAT": "gatv2", "GATE": "gatv2e"}
# Evaluation
KS = (1, 5, 10, 50)

all_pairs = df_node_alignment_truth.values.astype(np.int64)   # [K, 2]: (nodo en grafo 1, nodo en grafo 2)
if USE_VALIDATION:
	VAL_PAIRS, EVAL_PAIRS = make_val_split(all_pairs, val_frac=VAL_FRAC, seed=SEED)
else:
	VAL_PAIRS, EVAL_PAIRS = None, all_pairs
df_eval_truth = pd.DataFrame(EVAL_PAIRS)
print("Pares totales:", len(all_pairs), "| validación:", 0 if VAL_PAIRS is None else len(VAL_PAIRS),
	  "| evaluación:", len(EVAL_PAIRS))

Pares totales: 1570 | validación: 0 | evaluación: 1570


In [21]:
encoders

['GCNRes', 'GIN', 'GINE']

In [22]:
list_train_set

[['SecoAmazonas_1', 'SecoAmazonas_2'],
 ['FrescoAmazonas_1', 'FrescoAmazonas_2']]

In [23]:
list_train_loader

[{'SecoAmazonas_1': Data(x=[1570, 5], edge_index=[2, 303748], edge_attr=[303748, 3], edge_weight=[303748]),
  'SecoAmazonas_2': Data(x=[1570, 5], edge_index=[2, 287500], edge_attr=[287500, 3], edge_weight=[287500])},
 {'FrescoAmazonas_1': Data(x=[1570, 5], edge_index=[2, 274182], edge_attr=[274182, 3], edge_weight=[274182]),
  'FrescoAmazonas_2': Data(x=[1570, 5], edge_index=[2, 301072], edge_attr=[301072, 3], edge_weight=[301072])}]

In [ ]:
# [MEJORA 4] Un solo modelo parametrizado (model_PyG.TGAE) en lugar de una clase por encoder
list_runtime = []
list_fit_info = []

for encoder in encoders:
	for i, train_loader in enumerate(list_train_loader):
		group_id = groups_id[i]
		subgroup_id = subgroups_id[group_id]
		train_set = list_train_set[i]

		no_samples = len(train_set)
		input_dim = train_loader[train_set[0]].num_node_features
		edge_dim = train_loader[train_set[0]].edge_attr.size(1) if encoder in ("GINE", "GATE") else None

		# [MEJORA 4a] Semilla ANTES de construir el modelo: antes cada encoder continuaba el flujo aleatorio
		# del anterior (resultados dependientes del orden de `encoders`) y los pesos iniciales no se repetían.
		set_seed(SEED)
		model = TGAE(ENCODER_TO_CONV[encoder], NUM_HIDDEN_LAYERS, input_dim, HIDDEN_DIM,
					 output_feature_size, edge_dim=edge_dim, heads=HEADS).to(device)

		print("Fitting model")
		print(encoder, dataset, train_set, lr, epochs, input_dim, output_feature_size, no_samples)

		# [MEJORA 4b] CUDA es asíncrono: sincronizar antes y después para medir el tiempo real.
		# Nota: con validación activa, el tiempo incluye las evaluaciones periódicas.
		if device.type == "cuda":
			torch.cuda.synchronize(device)
		start_time = time.perf_counter()

		info_fit = fit_TGAE_subgraph(
			model, encoder, dataset, group_id, subgroup_id, train_loader, train_set, no_samples,
			epochs, lr, WEIGHT_DECAY,
			val_pairs=VAL_PAIRS, eval_every=EVAL_EVERY, select_metric=SELECT_METRIC, retrieval=RETRIEVAL,
			decoder=DECODER, tau=TAU, neg_ratio=NEG_RATIO, aug_p=AUG_P, grad_clip=GRAD_CLIP,
			patience=PATIENCE, warmup=WARMUP,
		)

		if device.type == "cuda":
			torch.cuda.synchronize(device)
		end_time = time.perf_counter()
		execution_time = end_time - start_time
		
		list_runtime.append(execution_time)
		list_fit_info.append({"encoder": encoder, "group_id": group_id,
							  **{k: v for k, v in info_fit.items() if k not in ("loss", "val_history")}})

		# [MEJORA 4c] Liberar el modelo (12 capas x 1024) antes del siguiente encoder
		del model
		if device.type == "cuda":
			torch.cuda.empty_cache()

df_fit = pd.DataFrame(list_fit_info)
df_fit

Fitting model
GCNRes exp110 ['SecoAmazonas_1', 'SecoAmazonas_2'] 0.0001 500 5 128 2


Epoch: 54:  11%|█         | 53/500 [00:56<08:00,  1.08s/it, Loss: 0.6730]


Early stopping (pérdida) en la época 54
Fitting model
GCNRes exp110 ['FrescoAmazonas_1', 'FrescoAmazonas_2'] 0.0001 500 5 128 2


Epoch: 55:  11%|█         | 54/500 [00:49<06:45,  1.10it/s, Loss: 0.6737]


Early stopping (pérdida) en la época 55
Fitting model
GIN exp110 ['SecoAmazonas_1', 'SecoAmazonas_2'] 0.0001 500 5 128 2


Epoch: 74:  15%|█▍        | 73/500 [01:14<07:18,  1.03s/it, Loss: 0.6932]


Early stopping (pérdida) en la época 74
Fitting model
GIN exp110 ['FrescoAmazonas_1', 'FrescoAmazonas_2'] 0.0001 500 5 128 2


Epoch: 74:  15%|█▍        | 73/500 [01:02<06:08,  1.16it/s, Loss: 0.6932]


Early stopping (pérdida) en la época 74
Fitting model
GINE exp110 ['SecoAmazonas_1', 'SecoAmazonas_2'] 0.0001 500 5 128 2


Epoch: 235:  47%|████▋     | 234/500 [05:56<08:45,  1.98s/it, Loss: 0.5977]